<a href="https://colab.research.google.com/github/fuenmayorscarlet610-dotcom/Aegis-Report-System./blob/main/Radar_Caribe_Vigilante.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install websockets folium

In [ ]:
# @title
import asyncio
import websockets
import json
import folium
import requests
from IPython.display import display, clear_output

# --- CREDENCIALES DE SEGURIDAD ---
AUTORA = "Scarlet Fuenmayor"
SISTEMA = "UNIDAD DE VIGILANCIA FUENMAYOR - AIS REAL-TIME"
MI_LLAVE = "913e893de112a3a24210c87aa89e2fd3e105d1c4"
TELEGRAM_TOKEN = "8568389728:AAEh_psJHUQGMFNpfNjCsBmgA4Wo6TCJRnU"
MI_CHAT_ID = "8144819503"

def enviar_alerta_tecnica(datos):
    url = f"https://api.telegram.org/bot{TELEGRAM_TOKEN}/sendMessage"
    mensaje = (f"🚢 {SISTEMA}\n"
               f"----------------------------\n"
               f"VESSEL: {datos['nombre']}\n"
               f"DESDESTINODESDESTINOTINODESDESTINODESDESTINOTINOTINO TINO: {datos['destino']}\n"
               f"LAT/LON: {datos['lat']}, {datos['lon']}\n"
               f"AUTORÍA: {AUTORA}\n"
               f"----------------------------")
    requests.post(url, json={"chat_id": MI_CHAT_ID, "text": mensaje})

async def radar_fuenmayor_v2():
    url = "wss://stream.aisstream.io/v0/stream"
    # Mapa técnico oscuro
    mapa = folium.Map(location=[10.6, -66.9], zoom_start=9, tiles='CartoDB dark_matter')

    # Sello de autoría en el mapa
    html_sello = f'''
        <div style="position: fixed; top: 10px; right: 10px; width: 250px; z-index:9999;
        background-color: #000; color: #0f0; border: 2px solid #0f0;
        font-family: monospace; padding: 10px; font-size: 11px;">
        <b>RADAR ESTRATÉGICO</b><br>
        ANÁLISIS: {AUTORA}<br>
        SISTEMA: VINCULADO SATÉLITE AIS
        </div>
    '''
    mapa.get_root().html.add_child(folium.Element(html_sello))

    async with websockets.connect(url) as websocket:
        # Área que cubre desde Puerto Cabello hasta Higuerote (como tu foto)
        sub = {"APIKey": MI_LLAVE, "BoundingBoxes": [[[10.0, -70.0], [13.0, -66.0]]]}
        await websocket.send(json.dumps(sub))

        print(f"🛰️ SISTEMA {AUTORA} INICIADO...")

        async for mensaje in websocket:
            msg = json.loads(mensaje)

            # Extraer especificaciones técnicas
            meta = msg.get('MetaData', {})
            ship_name = meta.get('ShipName', 'Desconocido')
            lat, lon = meta.get('latitude'), meta.get('longitude')

            # Datos adicionales para que sea "Real"
            # Nota: Algunos barcos no envían destino en cada segundo
            info_barco = {
                "nombre": ship_name,
                "lat": lat,
                "lon": lon,
                "destino": "En tránsito / Verificando..."
            }

            # Enviar notificación profesional
            enviar_alerta_tecnica(info_barco)

            # Dibujar en el mapa con icono de radar
            folium.CircleMarker(
                location=[lat, lon],
                radius=7,
                color='#0f0',
                fill=True,
                popup=f"Buque: {ship_name} | Analista: {AUTORA}"
            ).add_to(mapa)

            clear_output(wait=True)
            print(f"✅ RASTREO REAL: {ship_name} | Coord: {lat}, {lon}")
            display(mapa)

await radar_fuenmayor_v2()

✅ RASTREO REAL: AXION | Coord: 12.16413, -68.2853


CancelledError: 

In [ ]:
!npx localtunnel --port 8000